# Notebook verify — Module 2 (Reservoir Computing & QRC)

Verify nguyen ly Reservoir Computing (fixed reservoir + readout tuyen tinh) bang mot
Echo State Network CO DIEN thuc su chay duoc, tren chuoi NARMA-10 tong hop (dung benchmark
GOC cua paper QRC, khong phai du lieu khi hau).

CANH BAO: day la ESN CO DIEN (ma tran trong so ngau nhien thuc), KHONG phai mach luong tu
qiskit theo dung Hamiltonian MFIM (Eq. 1) cua paper arXiv:2608.28440v1 - minh hoa dung
NGUYEN TAC chung cua Reservoir Computing (reservoir co dinh, chi train readout).

In [1]:
import numpy as np
rng = np.random.default_rng(42)

def make_narma10(n, rng):
    u = rng.uniform(0, 0.5, n)
    y = np.zeros(n)
    for t in range(10, n):
        y[t] = (0.3*y[t-1] + 0.05*y[t-1]*np.sum(y[t-10:t])
                + 1.5*u[t-1]*u[t-10] + 0.1)
    return u, y

u, y = make_narma10(2000, rng)
print("NARMA-10 series, shape:", u.shape, y.shape, "range y:", y.min(), y.max())

NARMA-10 series, shape: (2000,) (2000,) range y: 0.0 0.8093707248879468


## Echo State Network (fixed reservoir, chi train readout)

Day la kien truc GOC (Jaeger 2001) ma QRC thay reservoir co dien bang tien hoa Hamiltonian
luong tu: reservoir KHONG BAO GIO duoc huan luyen bang gradient, chi lop readout tuyen tinh
phia sau duoc train (closed-form ridge regression) - dung nguyen tac nay la dieu can verify.

In [2]:
N_RES = 100
W_in = rng.uniform(-1, 1, N_RES)
W_res = rng.uniform(-1, 1, (N_RES, N_RES))
# chuan hoa spectral radius < 1 - DIEU KIEN BAT BUOC cho echo state property
eigvals = np.linalg.eigvals(W_res)
W_res = W_res / (np.max(np.abs(eigvals)) + 1e-9) * 0.9
print("Spectral radius sau chuan hoa:", np.max(np.abs(np.linalg.eigvals(W_res))))
assert np.max(np.abs(np.linalg.eigvals(W_res))) < 1.0, "Echo state property yeu cau spectral radius < 1"

states = np.zeros((len(u), N_RES))
s = np.zeros(N_RES)
for t in range(len(u)):
    s = np.tanh(W_in * u[t] + W_res @ s)
    states[t] = s
print("OK: reservoir co dinh da tien hoa", len(u), "buoc, KHONG co gradient nao duoc tinh cho W_in/W_res.")

Spectral radius sau chuan hoa: 0.8999999998535186
OK: reservoir co dinh da tien hoa 2000 buoc, KHONG co gradient nao duoc tinh cho W_in/W_res.


/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/1627443819.py:13: RuntimeWarning: divide by zero encountered in matmul
  s = np.tanh(W_in * u[t] + W_res @ s)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/1627443819.py:13: RuntimeWarning: overflow encountered in matmul
  s = np.tanh(W_in * u[t] + W_res @ s)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/1627443819.py:13: RuntimeWarning: invalid value encountered in matmul
  s = np.tanh(W_in * u[t] + W_res @ s)


In [3]:
# Readout tuyen tinh: ridge regression closed-form (giong cach train chuan cua MOI RC, ca QRC)
train_n = 1500
X_tr, y_tr = states[200:train_n], y[200:train_n]
X_te, y_te = states[train_n:], y[train_n:]
lam = 1e-6
W_out = np.linalg.solve(X_tr.T @ X_tr + lam*np.eye(N_RES), X_tr.T @ y_tr)
pred = X_te @ W_out
nrmse = np.sqrt(np.mean((pred - y_te)**2)) / np.std(y_te)
print(f"NRMSE tren test set NARMA-10 (ESN co dien, {N_RES} reservoir units): {nrmse:.4f}")
assert nrmse < 1.0, "Readout phai hoc duoc gi do (NRMSE < 1 nghia la tot hon du doan = trung binh)"
print("OK: readout tuyen tinh hoc duoc tu reservoir co dinh - dung nguyen tac RC ma QRC ke thua.")

NRMSE tren test set NARMA-10 (ESN co dien, 100 reservoir units): 0.3958
OK: readout tuyen tinh hoc duoc tu reservoir co dinh - dung nguyen tac RC ma QRC ke thua.


/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3456716193.py:6: RuntimeWarning: divide by zero encountered in matmul
  W_out = np.linalg.solve(X_tr.T @ X_tr + lam*np.eye(N_RES), X_tr.T @ y_tr)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3456716193.py:6: RuntimeWarning: overflow encountered in matmul
  W_out = np.linalg.solve(X_tr.T @ X_tr + lam*np.eye(N_RES), X_tr.T @ y_tr)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3456716193.py:6: RuntimeWarning: invalid value encountered in matmul
  W_out = np.linalg.solve(X_tr.T @ X_tr + lam*np.eye(N_RES), X_tr.T @ y_tr)
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3456716193.py:7: RuntimeWarning: divide by zero encountered in matmul
  pred = X_te @ W_out
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3456716193.py:7: RuntimeWarning: overflow encountered in matmul
  pred = X_te @ W_out
/var/folders/v0/j28fdbqj7bz6d2fbylr248x00000gn/T/ipykernel_76088/3